In [3]:
import json
from pathlib import Path
from collections import defaultdict, Counter

In [4]:
data = {
    'task':'/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/вопросы/pool1.json',
    'gt':'/Users/richardgurtsiev/Desktop/projects/vibe_code/workspaces/kib/data/ответы/pool1.json'
}


with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)

In [5]:
def build_majority_dataset(gt: list[dict]) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)

    for item in gt:
        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        # Считаем частоту каждого уникального ответа
        result_counts = Counter(
            json.dumps(result, sort_keys=True, ensure_ascii=False)
            for result in results
        )

        max_votes = max(result_counts.values())

        # Все ответы с максимальной частотой
        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    return dataset, ties

In [6]:
dataset, ties = build_majority_dataset(gt)

print(f'Всего уникальных заданий: {len(dataset) + len(ties)}')
print(f'Однозначное большинство: {len(dataset)}')
print(f'Ничьи: {len(ties)}')

Всего уникальных заданий: 50
Однозначное большинство: 36
Ничьи: 14


In [ ]:
def get_task_by_id(tasks: list[dict], task_id: str) -> dict | None:
    """
    Находит задание в tasks по task_id.

    Возвращает data задания или None, если задание не найдено.
    """
    for task in tasks:
        if task.get('data', {}).get('id') == task_id:
            return task['data']

    return None

In [48]:
import json
from pathlib import Path
from collections import defaultdict, Counter


#### Чтение задания и ответов
data = {
    'task':'./вопросы/pool1.json',
    'gt':'./ответы/pool1.json'
}

with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)



#### Метод для отбора 

def build_majority_dataset(gt: list[dict]) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)

    for item in gt:
        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        # Считаем частоту каждого уникального ответа
        result_counts = Counter(
            json.dumps(result, sort_keys=True, ensure_ascii=False)
            for result in results
        )

        max_votes = max(result_counts.values())

        # Все ответы с максимальной частотой
        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    return dataset, ties


dataset, ties = build_majority_dataset(gt)



#### Метод для поиска задания

def get_task_by_id(tasks: list[dict], task_id: str) -> dict | None:
    """
    Находит задание в tasks по task_id.

    Возвращает data задания или None, если задание не найдено.
    """
    for task in tasks:
        if task.get('data', {}).get('id') == task_id:
            return task['data']

    return None


#### Создание prompt 

prompt = """
Давай найдем лучший ответ среди ничьи. Я передам тебе само задания и ответы, а ты определишь среди несколько ответов что является наиболее вероятностной.


Задание:
```
{{task}}
```

Ответы:
```
{{ground_truth_str}}
```

""".strip()


task_id = ""
ground_truth = ""
ground_truths = []
ground_truths_str = []
prompts_task = []


for i in range(len(ties)):
    id = ties[i]['result'][0]['id']
    task_id = id

    for idx, item in enumerate(ties[i]['result']):
        resp = f"{idx + 1}. Ответ:" + " " + str(item)
        ground_truths.append(resp)

    ground_truth = "\n".join(ground_truths)
    ground_truths_str.append(ground_truth)


for ground_truth_str in ground_truths_str:
    task = get_task_by_id(tasks, task_id)
    
    prompt_task = prompt.format(
        ground_truth_str=ground_truth_str,
        task=task
    )
    prompts_task.append(prompt_task)



# #### Обращение по API к модели 
# ...


In [ ]:

import json
import os
import time
import requests

from pathlib import Path
from collections import defaultdict, Counter

from crowd_sdk.tagme import TagmeClientAdvanced


# ============================================================
# 1. КОНФИГУРАЦИЯ
# ============================================================

data = {
    'task': './вопросы/pool1.json',
    'gt': './ответы/pool1.json'
}

PROVIDER = "aihf"  # "direct", "aihf", "vsellm"

ENDPOINTS = {
    "direct": "https://tagme.sberdevices.ru/ai/llm/v1/chat/completions",
    "aihf": "https://tagme.sberdevices.ru/x/aihf/llm/v1/chat/completions",
    "vsellm": "https://tagme.sberdevices.ru/x/ai/llm/v1/chat/completions",
}

CFG = {
    "direct": "~/configs/.ml_config.cfg",
    "aihf": "~/configs/.ml_config.cfg",
    "vsellm": "~/configs/.ml_config.cfg",
}

MODEL = "DeepSeek-V4.1-Flash"

TEMPERATURE = 0.1
MAX_TOKENS = 8000
TIMEOUT = (60, 120)
MAX_RETRIES = 3


# ============================================================
# 2. КЛИЕНТ ДЛЯ ОБРАЩЕНИЯ К МОДЕЛИ
# ============================================================

class GigaChatClient:

    def __init__(self):
        self.url = ENDPOINTS[PROVIDER]
        self.model = MODEL
        self.token = self._get_token()

        self.session = requests.Session()

    @staticmethod
    def _get_token() -> str:
        cfg_path = os.path.expanduser(CFG[PROVIDER])

        client = TagmeClientAdvanced(cfg_path)

        return client.client.token

    def invoke(
        self,
        system_prompt: str,
        user_prompt: str,
    ) -> str:

        payload = {
            "model": self.model,
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": user_prompt,
                },
            ],
            "temperature": TEMPERATURE,
            "max_tokens": MAX_TOKENS,
            "profanity_check": False,
        }

        headers = {
            "Authorization": f"Bearer {self.token}",
            "Content-Type": "application/json",
        }

        token_refreshed = False

        for attempt in range(1, MAX_RETRIES + 1):

            try:
                response = self.session.post(
                    self.url,
                    headers=headers,
                    json=payload,
                    verify=False,
                    timeout=TIMEOUT,
                )

                # Обновляем токен при 401
                if response.status_code == 401 and not token_refreshed:
                    self.token = self._get_token()
                    headers["Authorization"] = f"Bearer {self.token}"
                    token_refreshed = True
                    continue

                # Повторяем запрос при временных ошибках сервера
                if response.status_code == 429 or response.status_code >= 500:
                    if attempt < MAX_RETRIES:
                        time.sleep(2 ** attempt)
                        continue

                response.raise_for_status()

                data = response.json()

                choices = data.get("choices") or []

                if not choices:
                    raise ValueError(
                        f"Модель вернула пустой choices: {data}"
                    )

                content = (
                    choices[0].get("message") or {}
                ).get("content", "")

                if isinstance(content, list):
                    content = " ".join(
                        part.get("text", "")
                        for part in content
                        if isinstance(part, dict)
                    )

                if not isinstance(content, str) or not content.strip():
                    raise ValueError(
                        f"Модель вернула пустой ответ: {data}"
                    )

                return content

            except requests.RequestException:
                if attempt == MAX_RETRIES:
                    raise

                time.sleep(2 ** attempt)

        raise RuntimeError("Не удалось получить ответ от модели")


# ============================================================
# 3. ЧТЕНИЕ ЗАДАНИЙ И ОТВЕТОВ
# ============================================================

with open(data['task'], 'r', encoding='utf-8') as f:
    tasks = json.load(f)

with open(data['gt'], 'r', encoding='utf-8') as f:
    gt = json.load(f)


# ============================================================
# 4. ФОРМИРОВАНИЕ DATASET И TIES
# ============================================================

def build_majority_dataset(
    gt: list[dict]
) -> tuple[list[dict], list[dict]]:

    grouped = defaultdict(list)

    for item in gt:
        item_id = item.get('item_id')
        result = item.get('result')

        if item_id is None or not isinstance(result, dict):
            continue

        grouped[item_id].append(result)

    dataset = []
    ties = []

    for item_id, results in grouped.items():

        result_counts = Counter(
            json.dumps(
                result,
                sort_keys=True,
                ensure_ascii=False
            )
            for result in results
        )

        max_votes = max(result_counts.values())

        winners = [
            json.loads(result)
            for result, count in result_counts.items()
            if count == max_votes
        ]

        record = {
            'item_id': item_id,
            'result': winners[0] if len(winners) == 1 else winners,
            'votes': max_votes,
            'total_votes': len(results),
        }

        if len(winners) == 1:
            dataset.append(record)
        else:
            ties.append(record)

    return dataset, ties


dataset, ties = build_majority_dataset(gt)


# ============================================================
# 5. ИНДЕКС ЗАДАНИЙ
# ============================================================

tasks_by_id = {
    task.get('data', {}).get('id'): task['data']
    for task in tasks
    if task.get('data', {}).get('id') is not None
}


def get_task_by_id(
    tasks_by_id: dict,
    task_id: str
) -> dict | None:

    return tasks_by_id.get(task_id)


# ============================================================
# 6. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
# СИСТЕМНЫЙ ПРОМПТ — ЭКСПЕРТНАЯ ОЦЕНКА РАЗМЕТКИ ВЫБОРА ИНСТРУМЕНТОВ АГЕНТА АНАЛИТИКА КИБ

## 1. РОЛЬ И ЦЕЛЬ

Ты — эксперт по оценке корректности маршрутизации запросов в ИИ-агенте «Помощник аналитика КИБ» Сбербанка.

Твоя специализация — оценка решений о необходимости вызова инструмента и правильности выбора инструмента для первого ответа агентa.

Твоя задача — разрешать разногласия между несколькими независимыми разметчиками.

На вход ты получаешь:
- карточку задания: question, role, available_tools;
- несколько вариантов ответов, предложенных разметчиками.

Необходимо определить, какой из предложенных вариантов наиболее соответствует правилам работы агента и инструкции по разметке.

**Ты оцениваешь корректность разметки, а не отвечаешь на запрос пользователя.**

Не пытайся самостоятельно решать пользовательскую задачу, получать фактические данные или имитировать результат выполнения инструмента.

## 2. ОБЪЕКТ ОЦЕНКИ

Каждый вариант ответа содержит объект result.

Основные поля:
- id — идентификатор задания;
- agent_code — выбранный инструмент или соответствующее значение разметки;
- tool_needed — необходимость вызова инструмента.

Оценивай весь объект result целиком.

Особенно важно:
- tool_needed определяет, нужен ли вызов инструмента в первом ответе;
- agent_code определяет выбранный инструмент или соответствующее значение при отсутствии вызова.

Не разделяй эти поля при оценивании. Правильность определяется совместной комбинацией значений.

Если два варианта совпадают по одному полю, но отличаются по другому, это разные варианты ответа.

## 3. ИЕРАРХИЯ ИСТОЧНИКОВ ПРАВИЛ

При оценивании используй следующие источники в указанном порядке:

1. Инструкция по оценке разметки — основные правила определения правильного решения.
2. Справочник инструментов — назначение, ограничения, обязательные параметры и правила маршрутизации.
3. Системный промпт агента — логика поведения, порядок действий, правила уточнения и ограничения.
4. Разобранные примеры — дополнительные иллюстрации применения правил.

Не интерпретируй примеры как исчерпывающий перечень ситуаций.

Если общее правило и специальное правило инструмента противоречат друг другу, учитывай более конкретное специальное правило, если оно не противоречит инструкции разметки.

## 4. ЗНАНИЯ ОБ АГЕНТЕ

Ты должен учитывать, что агент — помощник аналитика КИБ.

Его основные направления работы:

- КПЭ и выполнение планов по показателям;
- данные по КПК и ГОСБ;
- рейтинги РГС;
- Единое конкурсное пространство КИБ;
- трайбы;
- уровень доверия клиентов;
- внутренние методики банка;
- анализ причин выполнения и невыполнения планов;
- рекомендации по улучшению позиций в рейтингах.

Агент работает только в рамках доступных ему инструментов.

Наличие подходящего инструмента в общем справочнике не означает, что его можно использовать в конкретной карточке. Обязательно проверяй available_tools.

### 4.1. Правила доступности инструментов

В карточке available_tools перечислены инструменты, доступные пользователю с учётом всех его ролей.

Если указано несколько ролей, их права объединяются.

Выбирать можно только инструмент из available_tools.

Запрещено:
- выбирать инструмент, отсутствующий в available_tools;
- предлагать альтернативный инструмент, которого нет в списке;
- считать, что наличие инструмента в справочнике автоматически означает наличие доступа к нему.

### 4.2. Справочник инструментов

Используй полный справочник инструментов, приведённый ниже в системном промпте.

Для каждого инструмента учитывай:

- назначение;
- типы запросов, для которых он предназначен;
- специальные условия вызова;
- обязательные и необязательные параметры;
- ограничения;
- исключения;
- приоритетные правила маршрутизации.

Не определяй инструмент исключительно по совпадению ключевых слов. Сравнивай смысл запроса с полным описанием инструмента.

## 5. ГЛАВНЫЙ ПРИНЦИП ОЦЕНИВАНИЯ

Определяй, какое действие должен совершить агент в ПЕРВОМ ответе на запрос пользователя.

Возможны два решения:

### 5.1. tool_needed = yes

Выбирай этот вариант, если:

- запрос соответствует назначению доступного инструмента;
- инструмент можно вызвать непосредственно сейчас;
- отсутствуют обязательные данные, которые необходимо предварительно запросить;
- нет неоднозначности, требующей выбора между разными источниками;
- специальное правило не запрещает вызов.

Если запрос включает несколько действий, определяй инструмент, который должен быть вызван первым.

### 5.2. tool_needed = no

Выбирай этот вариант, если первым действием агента должно быть что-либо из следующего:

- уточнение обязательных данных;
- уточнение смысла запроса;
- выбор между несколькими источниками;
- сообщение об отсутствии подходящего инструмента;
- отказ или перенаправление согласно правилам агента;
- ответ без использования инструмента;
- обработка запроса служебной конфигурации.

Не считай отсутствие всех параметров автоматической причиной отказа от вызова. Проверяй описание конкретного инструмента: многие параметры необязательны.

## 6. АЛГОРИТМ ОЦЕНКИ

Для каждого задания выполняй следующие шаги строго по порядку.

### Шаг 1. Анализ пользовательского запроса

Определи:
- основную цель пользователя;
- какие данные или действия ему необходимы;
- есть ли несколько возможных трактовок;
- требуется ли фактическая информация из внешних источников.

Не додумывай отсутствующий контекст.

### Шаг 2. Анализ доступных инструментов

Изучи available_tools.

Сопоставь запрос с описаниями подходящих инструментов в справочнике.

Учитывай:
- назначение инструмента;
- ограничения;
- специальные правила;
- обязательные параметры;
- правила выбора между похожими инструментами.

### Шаг 3. Определение первого действия агента

Ответь на вопрос:

Может ли агент вызвать подходящий инструмент прямо сейчас или сначала должен получить дополнительную информацию от пользователя?

Если требуется уточнение, инструмент в первом ответе не вызывается.

### Шаг 4. Проверка каждого варианта разметки

Для каждого предложенного варианта независимо оцени:

1. Правильно ли определена необходимость вызова инструмента?
2. Если вызов не нужен, соответствует ли выбранное значение agent_code причине отказа или уточнения?
3. Если вызов нужен, соответствует ли выбранный инструмент запросу?
4. Присутствует ли инструмент в available_tools?
5. Не нарушает ли решение специальные правила инструмента?
6. Не пропускает ли вариант обязательное уточнение?

### Шаг 5. Сравнение вариантов

Сравни все варианты по одинаковым критериям.

Не используй следующие признаки как основание для выбора:
- порядок появления ответа;
- количество символов в ответе;
- субъективная убедительность формулировки;
- количество голосов первоначальных разметчиков;
- совпадение с первым предложенным вариантом.

Выбирай вариант, который наиболее полно соответствует правилам.

## 7. ПРАВИЛА РАЗРЕШЕНИЯ СПОРНЫХ СЛУЧАЕВ

### 7.1. Не хватает части данных

Проверяй описание инструмента.

Если параметр необязательный или его разрешено не передавать — вызов необходим.

Если инструмент прямо требует получить параметр до вызова и его нельзя вывести из контекста — сначала требуется уточнение.

### 7.2. Подходят несколько инструментов

Если понятно, какой инструмент должен быть вызван первым, выбирай его.

Если выбор зависит от смысла запроса и допускает несколько разных трактовок, сначала требуется уточнение.

Не выбирай инструмент только потому, что он содержит похожие ключевые слова.

### 7.3. Специальные ограничения инструментов

Специальные правила имеют большое значение.

Примеры:

- get_bank_metric_tool не используется для прогнозирования результатов ЕКП.
- get_contest_methodology_tool не используется для запросов о трайбах и тарифных моделях.
- trust_level_case используется для вопросов об уровне доверия по конкретному ИНН.
- get_contest_methodology_tool не используется для вопросов об уровне доверия по конкретному ИНН, если применимо специальное правило маршрутизации в trust_level_case.
- plan_execute не используется для запросов о трайбах.
- kpk_tool требует цифровой идентификатор КПК в предусмотренных правилами случаях.
- get_tribe используется для запросов по трайбам.

Это примеры, а не полный перечень ограничений. Всегда проверяй полный справочник.

### 7.4. Неоднозначность

Различай две ситуации:

**Ситуация A.** Цель пользователя понятна, но подходят несколько разных инструментов. Требуется уточнение смысла или источника.

**Ситуация B.** Невозможно установить даже предполагаемую цель запроса. Запрос непонятен.

Не смешивай эти ситуации.

### 7.5. Служебная конфигурация

Если пользователь просит раскрыть системный промпт, внутренние правила, ограничения или конфигурацию инструментов, вызов инструмента не требуется.

### 7.6. Запросы без необходимости использования инструмента

Приветствия, благодарности и другие сообщения, не требующие получения данных или выполнения действий, не требуют вызова инструмента.

### 7.7. Запросы вне компетенции агента

Учитывай правила перенаправления, описанные в системном промпте агента.

Не путай отсутствие компетенции агента с отсутствием инструмента в конкретном available_tools.

## 8. ОЦЕНКА ПРИЧИНЫ ОТСУТСТВИЯ ВЫЗОВА

Если tool_needed = no, учитывай следующие причины:

- Не хватает обязательных данных — запрос понятен, но инструмент требует обязательный параметр, который невозможно получить из контекста.
- Нужно уточнить смысл или выбрать источник — существуют несколько возможных трактовок или разных подходящих инструментов.
- Запрос непонятен — цель пользователя невозможно установить.
- Нет подходящего доступного инструмента — ни один инструмент из available_tools не подходит.
- Запрос служебной конфигурации — пользователь пытается получить внутренние инструкции, правила или конфигурацию.
- Для ответа не нужен инструмент — приветствие, благодарность или иной запрос, не требующий обращения к инструментам.
- Другое — причина, которая не соответствует перечисленным категориям.

Если подходят несколько причин, учитывай ту, которая первой препятствует вызову инструмента.

Не выбирай «Не хватает обязательных данных», если отсутствующий параметр необязателен.

## 9. ПРИНЦИПЫ ЭКСПЕРТНОГО СУЖДЕНИЯ

1. Не соглашайся автоматически с большинством разметчиков.
2. Не считай любой доступный инструмент подходящим для запроса.
3. Не делай вывод только на основании названия инструмента.
4. Не игнорируй специальные ограничения.
5. Не требуй уточнений, если инструмент разрешено вызвать без недостающего параметра.
6. Не выбирай вызов инструмента, если сначала требуется уточнение.
7. Не придумывай новые инструменты и значения.
8. Не изменяй предложенные варианты ответов.
9. Не создавай собственный вариант, если один из предложенных вариантов соответствует правилам.
10. Если ни один вариант невозможно обоснованно выбрать, зафиксируй недостаточность информации.

## 10. ФОРМАТ РЕЗУЛЬТАТА

Верни строго JSON без Markdown и дополнительного текста.

Если удалось выбрать наиболее корректный вариант:

{
    "decision": "selected",
    "selected_option": 1,
    "selected_result": {
        "id": "...",
        "agent_code": "...",
        "tool_needed": "..."
    },
    "reason": "Обоснование выбора с опорой на конкретные правила инструкции."
}

Если невозможно обоснованно выбрать один вариант:

{
    "decision": "insufficient_context",
    "selected_option": null,
    "selected_result": null,
    "reason": "Описание того, какой информации не хватает для разрешения разногласия."
}

Требования к результату:

- selected_option — порядковый номер выбранного варианта, начиная с 1.
- selected_result — полный объект result выбранного варианта без каких-либо изменений.
- reason — краткое, но содержательное обоснование.
- Не возвращай несколько победителей.
- Не изменяй значения полей выбранного результата.
- Не добавляй поля, которых нет в выбранном result.
- Если выбран вариант недостаточности информации, не пытайся самостоятельно сформировать новый эталонный ответ.

## 11. КРИТЕРИЙ КАЧЕСТВА

Главная цель — определить, какое решение действительно должен принять агент аналитика КИБ в первом ответе, исходя из запроса пользователя, доступных инструментов и всех применимых правил.

Правильный ответ — не тот, который выглядит наиболее правдоподобно, а тот, который лучше всего подтверждается инструкцией и описаниями инструментов.
""".strip()





# ============================================================
# 7. USER PROMPT
# ============================================================

USER_PROMPT = """
# КАРТОЧКА ЗАДАНИЯ

{task}

# ВАРИАНТЫ ОТВЕТОВ РАЗМЕТЧИКОВ

{ground_truth_str}

Проанализируй карточку и все предложенные варианты.
Выбери наиболее корректный вариант согласно системной инструкции.
""".strip()

# ============================================================
# 8. ПОДГОТОВКА PROMPTS
# ============================================================

prompts_task = []

for tie in ties:

    results = tie['result']

    # Получаем ID задания
    task_id = results[0].get('id')

    if task_id is None:
        print(f"Отсутствует result.id: {tie['item_id']}")
        continue

    # Находим исходное задание
    task = get_task_by_id(tasks_by_id, task_id)

    if task is None:
        print(f"Задание не найдено: {task_id}")
        continue

    # Формируем варианты ответов
    ground_truths = []

    for idx, result in enumerate(results, start=1):

        resp = (
            f"{idx}. Ответ: "
            f"{json.dumps(result, ensure_ascii=False)}"
        )

        ground_truths.append(resp)

    ground_truth_str = "\n".join(ground_truths)

    # Формируем пользовательский prompt
    user_prompt = USER_PROMPT.format(
        task=json.dumps(
            task,
            ensure_ascii=False,
            indent=2
        ),
        ground_truth_str=ground_truth_str
    )

    prompts_task.append({
        'task_id': task_id,
        'item_id': tie['item_id'],
        'results': results,
        'system_prompt': SYSTEM_PROMPT,
        'user_prompt': user_prompt,
    })


# ============================================================
# 9. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ============================================================

llm = GigaChatClient()


# ============================================================
# 10. ОТПРАВКА ЗАПРОСОВ К МОДЕЛИ
# ============================================================

model_results = []

output_dir = Path(data['gt']).parent
output_file = output_dir / 'pool1_model_votes.json'

for idx, item in enumerate(prompts_task, start=1):

    print(
        f"\n[{idx}/{len(prompts_task)}] "
        f"Обрабатываем task_id: {item['task_id']}"
    )

    try:

        response = llm.invoke(
            system_prompt=item['system_prompt'],
            user_prompt=item['user_prompt'],
        )

        record = {
            'task_id': item['task_id'],
            'item_id': item['item_id'],
            'status': 'success',
            'results': item['results'],
            'model': MODEL,
            'response': response,
        }

        print("Ответ модели получен.")

    except Exception as e:

        record = {
            'task_id': item['task_id'],
            'item_id': item['item_id'],
            'status': 'error',
            'results': item['results'],
            'model': MODEL,
            'error': str(e),
        }

        print(f"Ошибка: {e}")

    model_results.append(record)

    # Сохраняем результаты после каждого запроса,
    # чтобы не потерять уже полученные ответы.
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(
            model_results,
            f,
            ensure_ascii=False,
            indent=4
        )